In [ ]:
import numpy as np

class GRU:
    def __init__(self, input_size, hidden_size, learning_rate=0.01):
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.lr = learning_rate
        
        # ==================== 初始化参数 ====================
        # 使用 Xavier 初始化来保持梯度的稳定性
        std = 1.0 / np.sqrt(hidden_size)
        
        # 1. 更新门 (z) 参数
        #Wz: 输入到更新门的权重矩阵
        #Uz: 旧隐藏状态到更新门的权重矩阵
        #bz: 更新门的偏置
        self.Wz = np.random.uniform(-std, std, (input_size, hidden_size))# 输入到更新门的权重,这里是均匀分布初始化，均值为0，范围是[-std, std]
        self.Uz = np.random.uniform(-std, std, (hidden_size, hidden_size))
        self.bz = np.zeros((1, hidden_size))
        
        # 2. 重置门 (r) 参数
        self.Wr = np.random.uniform(-std, std, (input_size, hidden_size))
        self.Ur = np.random.uniform(-std, std, (hidden_size, hidden_size))
        self.br = np.zeros((1, hidden_size))
        
        # 3. 候选状态 (h_tilde) 参数
        self.Wh = np.random.uniform(-std, std, (input_size, hidden_size))
        self.Uh = np.random.uniform(-std, std, (hidden_size, hidden_size))
        self.bh = np.zeros((1, hidden_size))
        
        # 用于参数更新的梯度缓存
        self.reset_grads()

    def sigmoid(self, x):
        return 1 / (1 + np.exp(-x))
    
    def d_sigmoid(self, sigmoid_out):
        # Sigmoid 导数: y * (1 - y)
        return sigmoid_out * (1 - sigmoid_out)

    def tanh(self, x):
        return np.tanh(x)
    
    def d_tanh(self, tanh_out):
        # Tanh 导数: 1 - y^2
        return 1 - tanh_out ** 2

    def reset_grads(self):
        """每轮反向传播前清空梯度"""
        self.dWz, self.dUz, self.dbz = np.zeros_like(self.Wz), np.zeros_like(self.Uz), np.zeros_like(self.bz)
        self.dWr, self.dUr, self.dbr = np.zeros_like(self.Wr), np.zeros_like(self.Ur), np.zeros_like(self.br)
        self.dWh, self.dUh, self.dbh = np.zeros_like(self.Wh), np.zeros_like(self.Uh), np.zeros_like(self.bh)

    def forward(self, inputs, h_prev_init=None):
        """
        处理整个序列的前向传播
        inputs: (seq_len, batch_size, input_size)
        """
        seq_len, batch_size, _ = inputs.shape
        
        # 如果没提供初始隐藏状态，全0初始化
        if h_prev_init is None:
            h_prev = np.zeros((batch_size, self.hidden_size))
        else:
            h_prev = h_prev_init

        # 缓存每个时刻的状态，用于反向传播
        # cache 结构: 字典列表，每个元素对应一个时间步 t
        self.cache = [] 
        
        h_t = h_prev
        hs = {} # 存储所有时刻的隐藏状态输出
        hs[-1] = h_t
        
        for t in range(seq_len):
            x_t = inputs[t]
            
            # --- 1. 重置门 r_t ---
            r_t = self.sigmoid(np.dot(x_t, self.Wr) + np.dot(h_t, self.Ur) + self.br)
            
            # --- 2. 更新门 z_t ---
            z_t = self.sigmoid(np.dot(x_t, self.Wz) + np.dot(h_t, self.Uz) + self.bz)
            
            # --- 3. 候选状态 h_tilde ---
            # 重置门作用于旧状态
            h_reset = r_t * h_t
            h_tilde = self.tanh(np.dot(x_t, self.Wh) + np.dot(h_reset, self.Uh) + self.bh)
            
            # --- 4. 最终状态 h_t ---
            h_prev_t = h_t # 保存更新前的状态
            h_t = (1 - z_t) * h_prev_t + z_t * h_tilde
            
            # 保存结果和中间变量
            hs[t] = h_t
            self.cache.append({
                'x_t': x_t,
                'h_prev': h_prev_t, # t-1 时刻的 h
                'r_t': r_t,
                'z_t': z_t,
                'h_tilde': h_tilde,
                'h_reset': h_reset  # r_t * h_prev
            })
            
        return hs, h_t

    def backward(self, dh_from_next_layer):
        """
        反向传播 (BPTT)
        dh_from_next_layer: 来自上层（比如输出层）的梯度序列
                            字典格式 {t: gradient} 或者 数组
        """
        seq_len = len(self.cache)
        
        # 初始化流向“未来”的梯度 (最后一个时刻之后没有梯度传回来)
        dh_next = np.zeros((1, self.hidden_size)) 
        
        self.reset_grads()

        # === 时间反向循环 (T, T-1, ..., 0) ===
        for t in reversed(range(seq_len)):
            # 获取当前时刻的缓存
            vars = self.cache[t]
            x_t = vars['x_t']
            h_prev = vars['h_prev'] # h_{t-1}
            r_t = vars['r_t']
            z_t = vars['z_t']
            h_tilde = vars['h_tilde']
            h_reset = vars['h_reset']
            
            # 1. 汇总梯度：来自下一时刻的梯度 + 来自当前时刻输出层的梯度
            # 假设 dh_from_next_layer 是个字典，存了每个时刻的 loss 对 h_t 的导数
            dh_t = dh_from_next_layer[t] + dh_next
            
            # ===============================================
            #  对公式 h_t = (1 - z) * h_prev + z * h_tilde 求导
            # ===============================================
            
            # 对 z_t 求导 (经过 h_t)
            dz_t = dh_t * (h_tilde - h_prev) * self.d_sigmoid(z_t)
            
            # 对 h_tilde 求导 (经过 h_t)
            dh_tilde = dh_t * z_t * self.d_tanh(h_tilde)
            
            # 对 h_prev 求导 (直接路径部分，后面还要加上通过 r, z 回传的部分)
            dh_prev_running = dh_t * (1 - z_t)

            # ===============================================
            #  计算各权重矩阵的梯度
            # ===============================================
            
            # --- A. 处理 h_tilde 分支 ---
            # h_tilde = tanh(x*Wh + (r*h_prev)*Uh + bh)
            self.dWh += np.dot(x_t.T, dh_tilde)
            self.dUh += np.dot(h_reset.T, dh_tilde)
            self.dbh += np.sum(dh_tilde, axis=0, keepdims=True)
            
            # 反向传播到 h_reset = r_t * h_prev
            dh_reset = np.dot(dh_tilde, self.Uh.T)
            
            # 分解 h_reset 对 r_t 和 h_prev 的梯度
            dr_t_temp = dh_reset * h_prev
            dh_prev_running += dh_reset * r_t # 加上通过 h_tilde 回传给 h_prev 的梯度
            
            # --- B. 处理 Reset Gate (r_t) ---
            # r_t 经过 sigmoid
            dr_t = dr_t_temp * self.d_sigmoid(r_t)
            
            self.dWr += np.dot(x_t.T, dr_t)
            self.dUr += np.dot(h_prev.T, dr_t)
            self.dbr += np.sum(dr_t, axis=0, keepdims=True)
            
            # 加上通过 r_t 回传给 h_prev 的梯度
            dh_prev_running += np.dot(dr_t, self.Ur.T)
            
            # --- C. 处理 Update Gate (z_t) ---
            # 上面已经算过 dz_t 了，这里直接算权重
            self.dWz += np.dot(x_t.T, dz_t)
            self.dUz += np.dot(h_prev.T, dz_t)
            self.dbz += np.sum(dz_t, axis=0, keepdims=True)
            
            # 加上通过 z_t 回传给 h_prev 的梯度
            dh_prev_running += np.dot(dz_t, self.Uz.T)
            
            # --- 更新流向上一时刻的梯度 ---
            dh_next = dh_prev_running

        # 为了防止梯度爆炸 (Exploding Gradients)，这是 RNN 中必须的步骤
        for dparam in [self.dWz, self.dUz, self.dbz, self.dWr, self.dUr, self.dbr, self.dWh, self.dUh, self.dbh]:
            np.clip(dparam, -1, 1, out=dparam)

    def update_params(self):
        """使用简单的 SGD 更新参数"""
        for param, dparam in zip([self.Wz, self.Uz, self.bz, self.Wr, self.Ur, self.br, self.Wh, self.Uh, self.bh],
                                 [self.dWz, self.dUz, self.dbz, self.dWr, self.dUr, self.dbr, self.dWh, self.dUh, self.dbh]):
            param -= self.lr * dparam

# ==========================================
# 完整训练流程演示 (Mock Training)
# ==========================================

# 1. 设置数据
seq_len = 5
batch_size = 1
input_size = 4
hidden_size = 8

# 模拟输入序列 (时间步=5, batch=1, 特征=4)
inputs = np.random.randn(seq_len, batch_size, input_size)
# 模拟目标输出 (假设我们要让每个时刻的 h_t 接近某个随机值)
targets = np.random.randn(seq_len, batch_size, hidden_size)

# 2. 初始化模型
model = GRU(input_size, hidden_size, learning_rate=0.1)

print("=== 开始训练模拟 ===")
for epoch in range(10): # 训练 10 轮
    
    # --- 前向传播 ---
    hs, _ = model.forward(inputs)
    
    # --- 计算 Loss 和 初始梯度 ---
    loss = 0
    dh_grads = {}
    for t in range(seq_len):
        # 简单的 MSE Loss: L = 0.5 * (h_t - target)^2
        # dL/dh_t = h_t - target
        loss += 0.5 * np.sum((hs[t] - targets[t])**2)
        dh_grads[t] = hs[t] - targets[t]
        
    print(f"Epoch {epoch}, Loss: {loss:.4f}")
    
    # --- 反向传播 ---
    model.backward(dh_grads)
    
    # --- 更新参数 ---
    model.update_params()

print("=== 训练结束，Loss 应该下降 ===")能不能画出各层尺寸呢，神经网络结构

In [2]:
import torch
import torch.nn as nn

class GRUModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers=1):
        super(GRUModel, self).__init__()
        
        # 保存参数以便 init_hidden 使用
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        # =================================================
        # 1. 定义 GRU 层
        # =================================================
        self.gru = nn.GRU(
            input_size=input_size,    # 输入维度 (例如 4)
            hidden_size=hidden_size,  # 隐藏维度 (例如 8)
            num_layers=num_layers,    # 层数 (例如 1)
            batch_first=True          #让输入变成 (Batch, Seq, Feat)
        )
        
        # =================================================
        # 2. 定义全连接层 (Decoder)
        # =================================================
        # GRU 的输出是 hidden_size，我们需要把它映射回 output_size (字典大小)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x, h0=None):
        # x shape: (Batch, Seq, Input_Size) 因为我们开了 batch_first=True
        
        # --- A. 通过 GRU 层 ---
        # out: 包含序列中每个时间步的输出 (Batch, Seq, Hidden)
        # h_n: 只包含最后一个时间步的隐藏状态 (Layers, Batch, Hidden)
        out, h_n = self.gru(x, h0)
        
        # --- B. 通过全连接层 ---
        # 我们通常把 out 里的每个时间步都拿去预测
        # out.shape -> (Batch, Seq, Hidden)
        
        # PyTorch 的 Linear 层很智能，支持多维输入，它只会对最后一维操作
        # 所以直接传进去即可，不需要 view(-1) 拍扁 (除非为了兼容老版本习惯)
        final_out = self.fc(out) 
        
        # final_out shape: (Batch, Seq, Output_Size)
        return final_out, h_n

# ================= 测试代码 =================
# 假设参数
INPUT_SIZE = 4   # 字典大小 (a,b,c,d)
HIDDEN_SIZE = 8  # 记忆维度
OUTPUT_SIZE = 4  # 输出分类数 (通常等于 Input Size)
BATCH_SIZE = 2   # 一次喂 2 条数据
SEQ_LEN = 5      # 句子长度 5

# 1. 实例化模型
model = GRUModel(INPUT_SIZE, HIDDEN_SIZE, OUTPUT_SIZE)

# 2. 创建假数据 (Batch, Seq, Input)
inputs = torch.randn(BATCH_SIZE, SEQ_LEN, INPUT_SIZE) 

# 3. 前向传播
outputs, h_final = model(inputs)

print(f"输入尺寸: {inputs.shape}")        # [2, 5, 4]
print(f"输出尺寸: {outputs.shape}")       # [2, 5, 4] -> (Batch, Seq, Class)
print(f"最后记忆: {h_final.shape}")       # [1, 2, 8] -> (Layer, Batch, Hidden)

输入尺寸: torch.Size([2, 5, 4])
输出尺寸: torch.Size([2, 5, 4])
最后记忆: torch.Size([1, 2, 8])


In [3]:
import torch
import torch.nn as nn

# 1. 定义一个简单的 GRU
# input_size=4 (特征), hidden_size=8 (记忆)
gru = nn.GRU(input_size=4, hidden_size=8, batch_first=True)

# 2. 打印每一层的名字和参数量
print(f"{'层名称 (Name)':<20} | {'尺寸 (Shape)':<20} | {'参数个数 (Count)':<15}")
print("-" * 65)

total_params = 0
for name, param in gru.named_parameters():
    # param.numel() 获取张量中元素的总个数 (number of elements)
    count = param.numel()
    total_params += count
    print(f"{name:<20} | {str(list(param.shape)):<20} | {count:<15}")

print("-" * 65)
print(f"总参数量 (Total Parameters): {total_params}")
"""
Input-to-Hidden。对应公式里的 W (处理输入 x 的矩阵)
ih: Input-to-Hidden。对应公式里的 W (处理输入 x 的矩阵)。
hh: Hidden-to-Hidden。对应公式里的 U (处理上一步记忆 h_{t-1} 的矩阵)。
l0: Layer 0。第 0 层（如果你设置 num_layers=2，就会出现 l1）。
"""

层名称 (Name)           | 尺寸 (Shape)           | 参数个数 (Count)   
-----------------------------------------------------------------
weight_ih_l0         | [24, 4]              | 96             
weight_hh_l0         | [24, 8]              | 192            
bias_ih_l0           | [24]                 | 24             
bias_hh_l0           | [24]                 | 24             
-----------------------------------------------------------------
总参数量 (Total Parameters): 336


In [ ]:
import torch
import torch.nn as nn

class ExplainableGRU(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(ExplainableGRU, self).__init__()
        self.hidden_size = hidden_size
        
        # 1. 定义权重
        # GRU 只有 2 个门 (Reset, Update) + 1 个候选隐状态 (Candidate) = 3 部分
        # 所以输出维度是 3 * hidden_size
        self.weight_ih = nn.Linear(input_size, 3 * hidden_size)  # 输入 x 的权重
        self.weight_hh = nn.Linear(hidden_size, 3 * hidden_size) # 隐状态 h 的权重
        
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        batch_size, seq_len, _ = x.size()
        
        # GRU 只有隐状态 h，没有细胞状态 c
        h_t = torch.zeros(batch_size, self.hidden_size).to(x.device)
        
        # === 手动循环时间步 ===
        for t in range(seq_len):
            x_t = x[:, t, :]
            
            # 2. 预计算所有线性变换
            # 将 x 和 h 分别通过线性层
            x_gates = self.weight_ih(x_t)
            h_gates = self.weight_hh(h_t)
            
            # 3. 切分权重
            # 我们把结果切分成 3 份: 重置门(r), 更新门(z), 候选状态(n)
            # x_r, x_z, x_n 分别对应输入 x 对这三部分的贡献
            x_r, x_z, x_n = x_gates.chunk(3, 1)
            h_r, h_z, h_n = h_gates.chunk(3, 1)
            
            # === GRU 的核心门控逻辑 ===
            
            # A. 重置门 (Reset Gate)
            # 决定：在计算新候选状态时，要“无视”多少之前的隐状态？
            # 这里的逻辑主要是：如果 r_t 趋近 0，说明之前的 h_{t-1} 对当前计算无关，重置它。
            r_t = torch.sigmoid(x_r + h_r)
            
            # B. 更新门 (Update Gate)
            # 决定：要在多大程度上保留旧状态，还是更新为新状态？
            # 它相当于 LSTM 中“输入门”和“遗忘门”的结合体。
            z_t = torch.sigmoid(x_z + h_z)
            
            # C. 候选隐状态 (New Candidate Hidden State)
            # 注意这里！r_t (重置门) 在这里起作用了。
            # 它直接乘在 h_n 上，如果 r_t 是 0，旧的隐状态信息就被抹去了。
            n_t = torch.tanh(x_n + r_t * h_n)
            
            # D. 最终隐状态更新
            # 这是 GRU 最经典的公式：线性插值
            # h_t = (1 - z) * 新候选 + z * 旧状态
            # (注：PyTorch 官方公式是 (1-z)*n + z*h，有的论文是 z*n + (1-z)*h，只是定义反了，本质一样)
            h_t = (1 - z_t) * n_t + z_t * h_t
            
        # 取最后一个时间步
        out = self.fc(h_t)
        return out

# 测试
model = ExplainableGRU(input_size=10, hidden_size=20, output_size=5)
dummy_input = torch.randn(32, 5, 10)
print("GRU 输出形状:", model(dummy_input).shape)